# Generate from a Mork-2 checkpoint

Loads one checkpoint and continues ten prompts: Python, plain English, and short logic problems.

This checkpoint is a base model. It was trained to continue documents, so the samples below are completions of an opening, not instruction following. Change `CHECKPOINT` to try a `step_NNNNNN.pt` snapshot.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mork.checkpoint import load_model_for_inference
from mork.generate import generate
from mork.tokenizer import MorkTokenizer

## Checkpoint and prompts

In [ ]:
CHECKPOINT = ROOT / "checkpoints" / "latest.pt"
TOKENIZER = ROOT / "data" / "tokenizer.json"

MAX_NEW_TOKENS = 160
TEMPERATURE = 0.8
TOP_K = 50
SEED = 1337
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Prompts end on a word or a closing quote. The tokenizer merges a newline with
# the next line's indentation, and a space with the next word, so a trailing
# "\n" or " " leaves the model at a token boundary it rarely saw in training.
PROMPTS = [
    (
        "python",
        "def fibonacci(n):\n"
        '    """Return the nth Fibonacci number."""',
    ),
    (
        "python",
        "class Stack:\n"
        "    def __init__(self):",
    ),
    (
        "python",
        "import json\n"
        "from pathlib import Path\n"
        "\n"
        "def read_config(path):",
    ),
    (
        "python",
        "# Return True if the string is a palindrome, ignoring case and spaces.\n"
        "def is_palindrome(text):",
    ),
    (
        "english",
        "Photosynthesis is the process by which",
    ),
    (
        "english",
        "The lighthouse keeper had not seen a ship in three weeks. On Thursday morning,",
    ),
    (
        "english",
        "A computer stores numbers in binary. That means",
    ),
    (
        "logic",
        "All mammals breathe air. A whale is a mammal. Therefore,",
    ),
    (
        "logic",
        "There are three boxes. One contains only apples, one contains only oranges, "
        "and one contains both. Every label is wrong.",
    ),
    (
        "logic",
        "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
        "The ball costs",
    ),
]

## Load

In [ ]:
assert CHECKPOINT.exists(), f"checkpoint not found: {CHECKPOINT}"
assert TOKENIZER.exists(), f"tokenizer not found: {TOKENIZER}"

tokenizer = MorkTokenizer.load(TOKENIZER)
model, model_cfg, step, loss = load_model_for_inference(CHECKPOINT, device=DEVICE)
print(f"checkpoint: {CHECKPOINT}")
print(f"step {step:,}  loss {loss:.4f}  device {DEVICE}")
print(
    f"L{model_cfg.n_layer} D{model_cfg.n_embd} "
    f"context {model_cfg.block_size} vocab {model_cfg.vocab_size}"
)

checkpoint: /home/mart/projects/llms/mork-2/checkpoints/latest.pt
step 18,199  loss 2.3597  device cuda
L12 D768 context 512 vocab 16384


## Samples

Each sample uses `SEED` plus its index, so a re-run with the same settings repeats. The printout separates the prompt from the generated continuation. Generation stops on `<|end|>` or after `MAX_NEW_TOKENS` new tokens.

In [ ]:
for i, (kind, prompt) in enumerate(PROMPTS, start=1):
    torch.manual_seed(SEED + i)
    if DEVICE == "cuda":
        torch.cuda.manual_seed(SEED + i)

    text, stats = generate(
        model,
        tokenizer,
        prompt,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_k=TOP_K,
        device=DEVICE,
    )
    generated = text[len(prompt):] if text.startswith(prompt) else text
    print(f"===== {i}. {kind} =====")
    print("--- prompt ---")
    print(prompt, end="" if prompt.endswith("\n") else "\n")
    print("--- generated ---")
    print(generated)
    print(
        f"[{stats['new_tokens']} new tokens | "
        f"{stats['elapsed_ms']:.0f} ms | "
        f"{stats['tokens_per_sec']:.1f} tok/s]"
    )
    print()

===== 1. python =====
--- prompt ---
def fibonacci(n):
    """Return the nth Fibonacci number."""
--- generated ---

    if n == 1:
        return 1
    else:
        return fibonacci(n-1) + fibonacci(n-2)

print(fibonacci(5))
'''
#4. Given a list of numbers, return True if the numbers are completely odd,
# False otherwise.
def isPodd(n):
    """Does n number in list make a list with even number."""
    if n % 2 == 0:
        return True
    else:
        return False

print(isPodd(10))
'''
#5. Given a string s, return True if it is a palindrome, or False otherwise.
def isPalindrome(s):
    """Return true if s is a palindrome."""
    s = s.
[160 new tokens | 4256 ms | 37.6 tok/s]

===== 2. python =====
--- prompt ---
class Stack:
    def __init__(self):
--- generated ---

        self.stack = []

    def __repr__(self):
        return "Stack({}, {})".format(self.stack.pop(), self.stack.pop())

    def __str__(self):
        return "Stack({}, {})".format(self.stack.pop(), self.stack.pop